# Sentinal-2 Land Imagery Classifier

In [39]:
from pathlib import Path

import torch
from torch.utils.data import DataLoader, Subset, random_split
from torchvision import transforms
from torchvision.datasets import EuroSAT
import torchvision.transforms.functional as TF

import random

In [40]:
DATA_ROOT = Path.cwd().parent / "data"

CLASS_NAMES = [ # ALPHABETICAL ORDER SAME AS TORCHVISION DATASET
    "AnnualCrop", "Forest", "HerbaceousVegetation", "Highway", "Industrial",
    "Pasture", "PermanentCrop", "Residential", "River", "SeaLake",
]

Per-channel (R, G, B) mean and std over the EuroSAT training data, in the 0-1
range that ToTensor() produces. Normalising centres each channel near zero,
which gives the optimiser an easier starting point and helps training converge faster. 
These values will be computed over the training set but are currently placeholders.

In [ ]:
# EUROSAT_MEAN = (0.3444, 0.3803, 0.4078)
# EUROSAT_STD = (0.2037, 0.1366, 0.1148)

Updated numbers after computing.

In [2]:
EUROSAT_MEAN = (0.3436920642852783, 0.37974652647972107, 0.40759754180908203)
EUROSAT_STD = (0.20261552929878235, 0.13720497488975525, 0.11578831076622009)


The EuroSAT dataset has 27,000 images, but neural networks have millions of parameters, enough to essentially memorise the training set rather than learn from it. This is overfitting, and it produces a model that scores well on images it has seen and poorly on anything new. I want the model to recognise general features, not specific training images. To fight overfitting I use data augmentation. Because Sentinel-2 tiles are nadir (top-down) images with no inherent orientation, a flipped or rotated forest is still a valid forest, so I can apply horizontal/vertical flips and small rotations to generate realistic new views of each image every epoch, without ever creating unrealistic samples. I intentionally avoided colour-based augmentation, since spectral colour is a primary class signal in EuroSAT (water is blue, vegetation green), and distorting it would create mislabeled examples.

In [42]:
class RandomRotation90:
    def __call__(self, img):
        angle = random.choice([0, 90, 180, 270])
        return TF.rotate(img, angle)

def build_transforms(augment: bool = False):
    
    steps = []

    if augment:
        steps += [
            transforms.RandomHorizontalFlip(),
            transforms.RandomVerticalFlip(),
            RandomRotation90(),
        ]

    steps += [
        transforms.ToTensor(), # PIL (H,W,C) uint8 [0,255] -> tensor (C,H,W) float [0,1]
        transforms.Normalize(mean=EUROSAT_MEAN, std=EUROSAT_STD),
    ]

    return transforms.Compose(steps)

Lets each split carry its own transform, so I can augment train but not val/test.

In [43]:
class _TransformedSubset(torch.utils.data.Dataset):
    
    def __init__(self, subset, transform):
        self.subset = subset
        self.transform = transform

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, index):
        image, label = self.subset[index]   
        if self.transform is not None:
            image = self.transform(image)
        return image, label

Splits the Dataset into 3 seed-reproducible sets (training, validation, and testing) in a 70/15/15 split, with excess going to the training set. <br>
Each set has its own distinct purpose: <br>
* Training (70%) - The model learns from the training set. It sees them over and over and adjusts its weights.
* Validation (15%) - The model never trains on these. They are used to check accuracy during development to make decisions: which architecture, how many epochs, what learning rate. It's like a "practice exam".
* Testing (15%) - The model sees these only at the end, to provide an honest accuracy score free from training and tuning bias.

In [44]:
def get_datasets(root=DATA_ROOT, seed=42, download=True, augment_train=False):
    root = Path(root)
    root.mkdir(parents=True, exist_ok=True)

    base = EuroSAT(root=str(root), download=download, transform=None)

    n_total = len(base)
    n_val = int(0.15 * n_total)
    n_test = int(0.15 * n_total)
    n_train = n_total - n_val - n_test

    generator = torch.Generator().manual_seed(seed)
    train_sub, val_sub, test_sub = random_split(
        base, [n_train, n_val, n_test], generator=generator
    )

    train_ds = _TransformedSubset(train_sub, build_transforms(augment=augment_train))
    val_ds   = _TransformedSubset(val_sub,   build_transforms(augment=False))
    test_ds  = _TransformedSubset(test_sub,  build_transforms(augment=False))
    return train_ds, val_ds, test_ds


def get_dataloaders(root=DATA_ROOT, seed=42, batch_size=64,
                    num_workers=0, download=True, augment_train=False):
    train_ds, val_ds, test_ds = get_datasets(
        root=root, seed=seed, download=download, augment_train=augment_train
    )

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  num_workers=num_workers)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, num_workers=num_workers)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False, num_workers=num_workers)
    return train_loader, val_loader, test_loader

Computes the true per-channel mean and standard deviation for normalisation, measured over the training set only. I compute these on the training split rather than the whole dataset so that no statistics from the validation or test images leak into the model's preprocessing, which keeps the test set a fully honest measure of performance. This runs once, and the results are pasted in as the EUROSAT_MEAN and EUROSAT_STD constants so they don't need recomputing on every run.

In [45]:
def compute_channel_stats(root=DATA_ROOT, seed=42):
    root = Path(root)
    base = EuroSAT(root=str(root), download=True, transform=transforms.ToTensor())

    n_total = len(base)
    n_val = int(0.15 * n_total)
    n_test = int(0.15 * n_total)
    n_train = n_total - n_val - n_test

    generator = torch.Generator().manual_seed(seed)
    train_sub, _, _ = random_split(base, [n_train, n_val, n_test], generator=generator)

    n_pixels = 0
    channel_sum = torch.zeros(3)
    channel_sum_sq = torch.zeros(3)
    for image, _ in train_sub:                      
        n_pixels += image.shape[1] * image.shape[2]
        channel_sum += image.sum(dim=(1, 2))        
        channel_sum_sq += (image ** 2).sum(dim=(1, 2))

    mean = channel_sum / n_pixels
    std = (channel_sum_sq / n_pixels - mean ** 2).sqrt()
    return tuple(mean.tolist()), tuple(std.tolist())

mean, std = compute_channel_stats()
print("mean:", mean)
print("std: ", std)

mean: (0.3436920642852783, 0.37974652647972107, 0.40759754180908203)
std:  (0.20261552929878235, 0.13720497488975525, 0.11578831076622009)


A self-check that builds the loaders and verifies the pipeline: split sizes (18900 / 4050 / 4050), batch shape (64, 3, 64, 64), and label shape. The printed pixel range confirms normalisation actually ran (roughly [-2, 2] rather than [0, 1]).

In [47]:
if __name__ == "__main__":
    train_loader, val_loader, test_loader = get_dataloaders(augment_train=True)

    print("Classes:", CLASS_NAMES)
    print("Train:", len(train_loader.dataset))   # expect 18900
    print("Val:  ", len(val_loader.dataset))     # expect 4050
    print("Test: ", len(test_loader.dataset))    # expect 4050

    images, labels = next(iter(train_loader))
    print("Batch images:", tuple(images.shape))  # expect (64, 3, 64, 64)
    print("Batch labels:", tuple(labels.shape))  # expect (64,)
    print("Pixel range: [%.3f, %.3f]" % (images.min(), images.max()))  # normalised, so roughly [-2, 2]

Classes: ['AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway', 'Industrial', 'Pasture', 'PermanentCrop', 'Residential', 'River', 'SeaLake']
Train: 18900
Val:   4050
Test:  4050
Batch images: (64, 3, 64, 64)
Batch labels: (64,)
Pixel range: [-2.267, 5.116]
